# 단계 4-5: 주가 변동성 계산 및 시각화

**목표**: yfinance로 주가 데이터 수집 → 변동성 계산(표준편차) → 시각화

**입력**: `data/processed/final_dataset_standardized.csv` + yfinance 주가 데이터

**출력**: `data/processed/final_with_volatility.csv` + 시각화

## 1. 필수 라이브러리 설치 및 임포트

In [ ]:
!pip install -q yfinance matplotlib seaborn pyyaml

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath('../..'))

from src.stock_analyzer import quarter_to_date, calculate_volatility, get_stock_ticker_mapping
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'DejaVu Sans'

print("✅ 모든 라이브러리 임포트 완료")

## 2. 설정 로드 및 데이터 불러오기

In [ ]:
config_path = os.path.join(os.path.dirname(__file__), '..', 'config.yaml')

with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# 데이터 로드
COMPANY_NAME = 'Samsung Electronics'  # 티커 맵핑에 사용
input_path = os.path.join(config['DATA_PATHS']['processed_data'], 'final_dataset_standardized.csv')

try:
    df = pd.read_csv(input_path, encoding='utf-8')
    print(f"✅ 데이터 로드 완료")
    print(f"   행 수: {len(df)}")
except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다: {input_path}")
    print("   먼저 03_data_preprocessing.ipynb을 실행해주세요.")

## 3. 분기 정보 파싱

In [ ]:
# year_quarter가 없으면 date에서 파싱
if 'year_quarter' not in df.columns:
    df['date'] = pd.to_datetime(df['date'], format='%Y%m%d', errors='coerce')
    df['year'] = df['date'].dt.year
    df['quarter'] = df['date'].dt.quarter
    df['year_quarter'] = df['year'].astype(str) + '-Q' + df['quarter'].astype(str)

# 분기 마지막 날짜 변환
df['quarter_end_date'] = df['year_quarter'].apply(quarter_to_date)

print("✅ 분기 정보 파싱 완료")
print(f"\n분기 범위:")
print(f"  시작: {df['quarter_end_date'].min()}")
print(f"  종료: {df['quarter_end_date'].max()}")

## 4. 주식 티커 매핑

In [ ]:
# 삼성전자 티커
ticker_map = get_stock_ticker_mapping()
tickey = ticker_map.get('삼성전자', '005930.KS')

print(f"✅ 주식 티커 확인")
print(f"   삼성전자: {ticker}")

## 5. 주가 변동성 계산

In [ ]:
print("🔄 주가 변동성 계산 중...")

# 테스트: 처음 5개 행만
df_test = df.head(5).copy()
volatilities = []

for idx, row in tqdm(df_test.iterrows(), total=len(df_test)):
    quarter_end = row['quarter_end_date']
    
    if pd.isna(quarter_end):
        volatilities.append(np.nan)
        continue
    
    # 분기 마지막 날로부터 1개월 후를 t+1
    end_date = quarter_end + pd.DateOffset(months=1)
    start_date = quarter_end
    
    vol = calculate_volatility(
        ticker,
        start_date.strftime('%Y-%m-%d'),
        end_date.strftime('%Y-%m-%d')
    )
    volatilities.append(vol)

df_test['stock_volatility'] = volatilities

print("✅ 변동성 계산 완료")
print(f"\n변동성 통계:")
print(df_test[['year_quarter', 'quarter_end_date', 'stock_volatility']].head(10))

## 6. 결과 저장

In [ ]:
# 테스트 결과만 저장
output_path = os.path.join(config['DATA_PATHS']['reports'], 'volatility_analysis_sample.csv')
os.makedirs(os.path.dirname(output_path), exist_ok=True)

df_test.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✅ 저장 완료: {output_path}")

## 7. 시각화 (점수 vs 변동성)

In [ ]:
# 산점도 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 확신도 vs 변동성
axes[0].scatter(df_test['확신도점수'], df_test['stock_volatility'], alpha=0.6, s=100)
axes[0].set_xlabel('Confidence Score')
axes[0].set_ylabel('Stock Volatility')
axes[0].set_title('Confidence Score vs Stock Volatility')
axes[0].grid(True, alpha=0.3)

# 강세전망 vs 변동성
axes[1].scatter(df_test['강세전망점수'], df_test['stock_volatility'], alpha=0.6, s=100, color='orange')
axes[1].set_xlabel('Bullish Outlook Score')
axes[1].set_ylabel('Stock Volatility')
axes[1].set_title('Bullish Outlook vs Stock Volatility')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(config['DATA_PATHS']['reports'], 'volatility_scatter.png'), dpi=100)
print("✅ 산점도 저장 완료")
plt.show()

## 8. 상관계수 계산

In [ ]:
# 수치형 컬럼만 선택
numeric_cols = df_test.select_dtypes(include=[np.number]).columns
corr_data = df_test[numeric_cols].dropna()

if len(corr_data) > 0:
    # 상관계수 계산
    corr_matrix = corr_data.corr()
    
    # 변동성과의 상관계수
    if 'stock_volatility' in corr_matrix.columns:
        volatility_corr = corr_matrix['stock_volatility'].sort_values(ascending=False)
        
        print("📊 변동성과의 상관계수:")
        print(volatility_corr)
        
        # 히트맵
        plt.figure(figsize=(10, 8))
        sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
        plt.title('Correlation Matrix')
        plt.tight_layout()
        plt.savefig(os.path.join(config['DATA_PATHS']['reports'], 'correlation_heatmap.png'), dpi=100)
        print("\n✅ 상관계수 히트맵 저장 완료")
        plt.show()

## ✅ 단계 4-5 완료!

전체 파이프라인 완료. 마지막으로 `00_main_workflow.ipynb`에서 전체 워크플로우를 확인하세요.